# Train issue-relation reranker v2

Use a **T4 GPU** and upload `relation_reranker_colab_v2_bundle.zip`. V2 uses a new family holdout, symmetric failure labels from v1, stronger exact-match ordering, and a small LoRA adapter. The exported adapter remains quarantined until the returned rankings receive symmetric top-10 review.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then reconnect.'
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, f'Upload exactly one bundle ZIP; received: {list(uploaded)}'
archive = archives[0]
run_dir = Path('/content/relation_reranker_v2')
shutil.rmtree(run_dir, ignore_errors=True)
run_dir.mkdir(parents=True)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(run_dir)
%cd /content/relation_reranker_v2

In [ ]:
# Colab currently preinstalls torchao 0.10, while recent PEFT rejects versions <=0.16.
# This model uses ordinary fp32/fp16 weights, so TorchAO is unnecessary.
!pip -q uninstall -y torchao
!pip -q install 'transformers>=4.44,<5' 'peft>=0.13' sentencepiece safetensors
!python train_relation_reranker.py --data data --out output --epochs 1 --batch-size 8 --grad-accum 2 --max-length 256 --lora --export-candidate

In [ ]:
import json
from pathlib import Path
report = json.loads(Path('output/report.json').read_text(encoding='utf-8'))
print('PRELIMINARY ACCEPTED:', report['accepted'])
print(json.dumps(report['gates'], indent=2, ensure_ascii=False))
print(json.dumps(report['production_baseline_vs_trained'], indent=2, ensure_ascii=False))

In [ ]:
!zip -qr relation_reranker_v2_result.zip output
from google.colab import files
files.download('relation_reranker_v2_result.zip')